In [0]:
import pandas as pd
import numpy as np
import time
import os

def generate_unified_massive_dataset(num_rows=1000000, save_directory=""):
    """
    Generates 1M rows using strict 35/25/20/20 weighted CIBIL scoring logic.
    """
    print(f"🚀 Starting generation of {num_rows:,} rows...")
    start_time = time.time()

    # ==========================================
    # 1-6. GENERATE ALL FEATURES (Vectorized)
    # ==========================================
    df = pd.DataFrame({
        'farmer_id': [f"ADHR-{np.random.randint(100000000, 999999999)}" for _ in range(num_rows)],
        'archetype': np.random.choice([0, 1, 2], p=[0.40, 0.45, 0.15], size=num_rows)
    })

    arch_0, arch_1, arch_2 = df['archetype'] == 0, df['archetype'] == 1, df['archetype'] == 2

    # Identity
    df['aadhaar_ekyc_verified'] = np.random.rand(num_rows) < 0.95
    df['agristack_id_linked'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.rand(num_rows) < 0.20, np.random.rand(num_rows) < 0.60, np.random.rand(num_rows) < 0.95])
    df['land_record_verified'] = np.where(arch_0, np.random.rand(num_rows) < 0.50, np.random.rand(num_rows) < 0.90)
    df['aadhaar_address_matches_mandi'] = np.where(arch_2, np.random.rand(num_rows) < 0.85, np.random.rand(num_rows) < 0.50)

    # Financial
    df['upi_monthly_avg_inr'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.uniform(500, 3000, num_rows), np.random.uniform(4000, 15000, num_rows), np.random.uniform(20000, 80000, num_rows)]).round(2)
    df['upi_transaction_count_30d'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.randint(1, 10, num_rows), np.random.randint(15, 40, num_rows), np.random.randint(50, 150, num_rows)])
    df['upi_merchant_vs_p2p_ratio'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.uniform(0.05, 0.20, num_rows), np.random.uniform(0.20, 0.50, num_rows), np.random.uniform(0.60, 0.95, num_rows)]).round(2)
    
    # Assets
    df['land_area_hectares'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.uniform(0.2, 0.9, num_rows), np.random.uniform(1.0, 3.5, num_rows), np.random.uniform(4.0, 15.0, num_rows)]).round(2)
    df['pmfby_insurance_enrolled'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.rand(num_rows) < 0.15, np.random.rand(num_rows) < 0.60, np.random.rand(num_rows) < 0.90])
    
    # Telecom & Misc
    df['telecom_recharge_streak_months'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.randint(0, 5, num_rows), np.random.randint(5, 25, num_rows), np.random.randint(24, 61, num_rows)])
    df['days_with_zero_balance_6m'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.randint(20, 90, num_rows), np.random.randint(2, 20, num_rows), np.random.randint(0, 2, num_rows)])

    # ==========================================
    # 7. FPO & SHG INTEGRATION
    # ==========================================
    df['is_fpo_shg_member'] = np.select([arch_0, arch_1, arch_2], 
        [np.random.rand(num_rows) < 0.20, np.random.rand(num_rows) < 0.55, np.random.rand(num_rows) < 0.85])

    rand_loan = np.random.rand(num_rows)
    cond_not_member = ~df['is_fpo_shg_member']
    cond_no_loan = df['is_fpo_shg_member'] & (rand_loan < 0.40)
    cond_default = df['is_fpo_shg_member'] & (rand_loan >= 0.40) & (
        (arch_0 & (rand_loan > 0.80)) | (arch_1 & (rand_loan > 0.90)) | (arch_2 & (rand_loan > 0.98)))
    cond_repaid = df['is_fpo_shg_member'] & (rand_loan >= 0.40) & ~cond_default

    df['fpo_shg_loan_repayment_history'] = np.select(
        [cond_not_member, cond_no_loan, cond_default, cond_repaid],
        ["Not Applicable", "No Loan Taken", "Defaulted", "Repaid On Time"],
        default="Not Applicable"
    )

    # ==========================================
    # 8. THE STRICT WEIGHTED TARGET CALCULATIONS
    # ==========================================
    print("⚖️ Applying strict 35/25/20/20 weights to CIBIL Score...")
    base_score = np.full(num_rows, 300.0)
    
    # 1. Repayment History (35% -> Max 210 points)
    # If repaid: full 210 pts. If no loan: 105 pts (neutral). If defaulted: 0 pts (massive hit).
    base_score += np.where(df['fpo_shg_loan_repayment_history'] == "Repaid On Time", 210, 0)
    base_score += np.where(df['fpo_shg_loan_repayment_history'].isin(["Not Applicable", "No Loan Taken"]), 105, 0)
    
    # 2. Financial Discipline (25% -> Max 150 points)
    # Split into three 50-point chunks based on the image features
    base_score += (df['upi_merchant_vs_p2p_ratio'] * 50) # Max 50 pts
    base_score += np.clip(df['telecom_recharge_streak_months'], 0, 50) # 1 pt per month, max 50 pts
    base_score += np.clip((df['upi_monthly_avg_inr'] / 40000) * 50, 0, 50) # Scales with income, max 50 pts
    
    # 3. Asset & Productivity (20% -> Max 120 points)
    base_score += np.where(df['land_record_verified'], 40, 0) # 40 pts
    base_score += np.where(df['pmfby_insurance_enrolled'], 40, 0) # 40 pts
    base_score += np.clip(df['land_area_hectares'] * 8, 0, 40) # 8 pts per hectare, max 40 pts
    
    # 4. Identity & Trust (20% -> Max 120 points)
    base_score += np.where(df['is_fpo_shg_member'], 40, 0) # 40 pts
    base_score += np.where(df['aadhaar_ekyc_verified'], 20, 0) # 20 pts
    base_score += np.where(df['agristack_id_linked'], 60, 0) # 60 pts (High trust signal)

    # Apply minor behavior penalties and noise
    base_score -= (df['days_with_zero_balance_6m'] * 0.5) 
    noise = np.random.randint(-25, 26, size=num_rows) # Lower noise because the rules are strict now
    
    # Finalize CIBIL
    df['synthetic_agri_cibil_score'] = np.clip(np.round(base_score + noise), 300, 900).astype(int)

    # Calculate final Loan Default (Target classification variable)
    default_prob = np.full(num_rows, 0.02)
    default_prob = np.where(df['synthetic_agri_cibil_score'] < 750, 0.08, default_prob)
    default_prob = np.where(df['synthetic_agri_cibil_score'] < 650, 0.25, default_prob)
    default_prob = np.where(df['synthetic_agri_cibil_score'] < 500, 0.85, default_prob)
    default_prob = np.where(df['fpo_shg_loan_repayment_history'] == "Defaulted", 0.95, default_prob) # The #1 Predictor
    
    df['historical_loan_default'] = np.random.rand(num_rows) < default_prob

    df = df.drop(columns=['archetype'])

    # ==========================================
    # 9. SAVE TO DATABRICKS DBFS
    # ==========================================
    if save_directory and not os.path.exists(save_directory):
        os.makedirs(save_directory, exist_ok=True)

    filename = f"master_hackathon_data_{num_rows}_rows.csv"
    full_path = os.path.join(save_directory, filename) if save_directory else filename

    print(f"💾 Saving to: {full_path} ... (This may take a few seconds)")
    df.to_csv(full_path, index=False)
    
    end_time = time.time()
    print(f"✅ Success! File saved at: {os.path.abspath(full_path)}")
    print(f"⏱️ Total execution time: {round(end_time - start_time, 2)} seconds")


# RUN THIS IN DATABRICKS (Change path if needed)
databricks_path = "/Workspace/Users/md25b045@smail.iitm.ac.in/Drafts/"
generate_unified_massive_dataset(num_rows=1000000, save_directory=databricks_path)

🚀 Starting generation of 1,000,000 rows...
⚖️ Applying strict 35/25/20/20 weights to CIBIL Score...
💾 Saving to: /Workspace/Users/md25b045@smail.iitm.ac.in/Drafts/master_hackathon_data_1000000_rows.csv ... (This may take a few seconds)
✅ Success! File saved at: /Workspace/Users/md25b045@smail.iitm.ac.in/Drafts/master_hackathon_data_1000000_rows.csv
⏱️ Total execution time: 11.4 seconds


In [0]:
import pandas as pd

# 1. Provide the exact path (No 'file:/' prefix needed for Pandas)
file_path = "/Workspace/Users/md25b045@smail.iitm.ac.in/Drafts/master_hackathon_data_1000000_rows.csv"

# 2. Read the CSV using Pandas (bypasses Spark's pathing issues)
print("Reading CSV via Pandas...")
pdf = pd.read_csv(file_path)

# 3. Convert the Pandas DataFrame into a PySpark DataFrame in memory
print("Converting to PySpark DataFrame...")
df = spark.createDataFrame(pdf)

# 4. Write it to the backend as a Delta Table
table_name = "default.agri_credit_features_1m"
print(f"Writing to Delta Table: {table_name}...")
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

print("✅ Data successfully ingested into Delta Lake!")

Reading CSV via Pandas...
Converting to PySpark DataFrame...
Writing to Delta Table: default.agri_credit_features_1m...
✅ Data successfully ingested into Delta Lake!


In [0]:
# Load the table
df = spark.table("default.agri_credit_features_1m")

# Drop the leaked CIBIL score column
df_cleaned = df.drop("synthetic_agri_cibil_score")

# Overwrite the table with the clean data
df_cleaned.write.format("delta").mode("overwrite").saveAsTable("default.agri_credit_features_1m")

print("✅ Data Leak Fixed! The 'synthetic_agri_cibil_score' has been removed.")

✅ Data Leak Fixed! The 'synthetic_agri_cibil_score' has been removed.


In [0]:
import pandas as pd

# 1. Path to your raw CSV
file_path = "/Workspace/Users/md25b045@smail.iitm.ac.in/Drafts/master_hackathon_data_1000000_rows.csv"

# 2. Read the CSV
print("Loading CSV...")
df = pd.read_csv(file_path)

# 3. Drop the column
df = df.drop(columns=['synthetic_agri_cibil_score'])

# 4. Overwrite the CSV file
print("Overwriting CSV...")
df.to_csv(file_path, index=False)

print("✅ Data Leak Fixed! Column removed from the raw CSV file too.")

Loading CSV...
Overwriting CSV...
✅ Data Leak Fixed! Column removed from the raw CSV file too.
